In [1]:
import sys

print("Python executable:")
print(sys.executable)

print("\nPython version:")
print(sys.version)
import sys

!{sys.executable} -m pip install tqdm

Python executable:
c:\Users\s2\Documents\FIXRKI\.fed\Scripts\python.exe

Python version:
3.11.9 (tags/v3.11.9:de54cf5, Apr  2 2024, 10:12:12) [MSC v.1938 64 bit (AMD64)]



[notice] A new release of pip is available: 24.0 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import tqdm
print(tqdm.__version__, tqdm.__file__)

4.70.1 c:\Users\s2\Documents\FIXRKI\.fed\Lib\site-packages\tqdm\__init__.py


In [1]:
import sys
 
print("Python executable:")
print(sys.executable)
 
print("\nPython version:")
print(sys.version)
import sys
 
!{sys.executable} -m pip install tqdm
 
# %%
import tqdm
print(tqdm.__version__, tqdm.__file__)
 
# %%
# ============================================================
# 0. IMPOR, LOGGING, QUIET, SELEKSI CLIENT
# ============================================================
import csv
import gc
import hashlib
import os
import pickle
import time
import logging
import contextlib
import importlib
from pathlib import Path
 
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from sklearn.metrics import (
    auc as sk_auc, roc_auc_score, average_precision_score,
    roc_curve, precision_recall_curve,
)
 
from dataset import prepare_client_data, TARGET, DROP_COLS
 
# Windows: start Ray kadang lambat/flaky saat di-restart berkali-kali.
# Perpanjang batas tunggu startup node (default 60 dtk).
os.environ.setdefault("RAY_NODE_START_TIMEOUT", "180")
 
# ------------------------------------------------------------
# >>> PILIH CLIENT YANG DIPAKAI <<<
#   SELECTION : "top10" | "top20" | "all"
# ------------------------------------------------------------
SELECTION = "top10"
 
# Urutan state berdasarkan rata-rata F1 test run sebelumnya (rank 1 -> 20)
RANKED_STATES = [
    54, 13, 23, 5, 29, 16, 41, 19, 10, 39,   # top 10
    30, 22, 55, 33, 34, 37, 50, 48, 12, 1,   # rank 11-20
]
TOP_N = {"top10": 10, "top20": 20}
 
VERBOSE = False
 
# Folder output terpisah per seleksi (otomatis memisahkan cache runs/*.pkl)
OUT_DIR = Path(f"hasil_fl_{SELECTION}")
OUT_DIR.mkdir(exist_ok=True)
LOG_FLWR = OUT_DIR / "flwrtop10personalized.log"
LOG_PRINTS = OUT_DIR / "printstop10personalized.log"
 
 
def setup_flwr_logging():
    """Console Flower hanya WARNING+; detail penuh ke file.
    Panggil SEBELUM tiap run (Flower bisa memasang ulang handler console)."""
    lg = logging.getLogger("flwr")
    lg.propagate = False
    for h in list(lg.handlers):
        if not isinstance(h, logging.FileHandler):
            h.setLevel(logging.WARNING)
    if not any(isinstance(h, logging.FileHandler) for h in lg.handlers):
        fh = logging.FileHandler(LOG_FLWR, encoding="utf-8")
        fh.setLevel(logging.INFO)
        fh.setFormatter(logging.Formatter("%(asctime)s %(levelname)s %(message)s"))
        lg.addHandler(fh)
    lg.setLevel(logging.INFO)
 
 
@contextlib.contextmanager
def quiet(logfile=None):
    """Alihkan print() (stdout) ke file. Dinonaktifkan bila VERBOSE.
    tqdm menulis ke stderr, jadi progress bar tetap tampil."""
    if VERBOSE:
        yield
        return
    with open(logfile or LOG_PRINTS, "a", encoding="utf-8") as f, \
            contextlib.redirect_stdout(f):
        yield
 
 
def run_quiet(fn, *args, **kwargs):
    """Jalankan fungsi training dengan log Flower dan print() dialihkan ke file."""
    setup_flwr_logging()
    with quiet():
        out = fn(*args, **kwargs)
    setup_flwr_logging()
    return out
 
 
setup_flwr_logging()

Python executable:
c:\Users\s2\Documents\FIXRKI\.fed\Scripts\python.exe

Python version:
3.11.9 (tags/v3.11.9:de54cf5, Apr  2 2024, 10:12:12) [MSC v.1938 64 bit (AMD64)]
4.70.1 c:\Users\s2\Documents\FIXRKI\.fed\Lib\site-packages\tqdm\__init__.py



[notice] A new release of pip is available: 24.0 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip
c:\Users\s2\Documents\FIXRKI\.fed\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:

# %%
# ============================================================
# 1. KONFIGURASI
# ============================================================
CLIENT_DIR = Path(r'C:\Users\s2\Documents\FIXRKI\bfprssprep\split\clients')
 
# ------------------------------------------------------------
# >>> PENGATURAN SEED (SATU TEMPAT UNTUK SEMUA) <<<
#   SINGLE_SEED : dipakai run tunggal (bagian 7 & 8).
#   SEEDS       : dipakai perbandingan multi-seed (bagian 9).
# ------------------------------------------------------------
SINGLE_SEED = 42
SEEDS = (42, 0, 123, 1337, 3407)
 
# ------------------------------------------------------------
# >>> PENGATURAN THRESHOLD (diteruskan ke fl_runner) <<<
#   THR_MODE : "fixed" | "per_client" | "global"
#              (per_client/global = threshold dipilih dari VAL, dipakai di TEST)
#   THR_CRIT : "f1" | "youden" | "gmean" | "sens_target"
# ------------------------------------------------------------
THR_MODE = "fixed"
THR_CRIT = "f1"
THR_KW = dict(threshold_mode=THR_MODE, threshold_criterion=THR_CRIT)
 
# ============================================================
# 2. MUAT DATA SEMUA CLIENT, LALU FILTER SESUAI SELECTION
# ============================================================
clients, scaler, feature_names = prepare_client_data(CLIENT_DIR)
 
 
def select_clients(clients, selection, ranked, top_n):
    """Ambil subset client sesuai ranking. Key dinormalisasi ke string
    (menangani key int 54, str "54", atau "state_54")."""
    if selection == "all":
        return dict(clients)
    if selection not in top_n:
        raise ValueError(f"SELECTION harus salah satu dari {list(top_n)} atau 'all'")
 
    by_str = {str(k).replace("state_", ""): k for k in clients}
    wanted = [str(s) for s in ranked[:top_n[selection]]]
 
    missing = [s for s in wanted if s not in by_str]
    if missing:
        print(f"[PERINGATAN] state tidak ditemukan di data: {missing}")
 
    return {by_str[s]: clients[by_str[s]] for s in wanted if s in by_str}
 
 
n_all = len(clients)
clients = select_clients(clients, SELECTION, RANKED_STATES, TOP_N)
assert len(clients) > 0, "Tidak ada client yang terpilih, periksa RANKED_STATES / key clients"
 
print(f"Seleksi       : {SELECTION} ({len(clients)} dari {n_all} client)")
print(f"Jumlah client : {len(clients)}")
print(f"Daftar state  : {list(clients.keys())}")
print(f"Target        : {TARGET}")
print(f"Kolom dibuang : {DROP_COLS}")

Seleksi       : top10 (10 dari 44 client)
Jumlah client : 10
Daftar state  : [54, 13, 23, 5, 29, 16, 41, 19, 10, 39]
Target        : _michd
Kolom dibuang : ['iyear', '_state', 'cvdinfr4', 'cvdcrhd4']


In [3]:

# ============================================================
# 3. DAFTAR FITUR
# ============================================================
n_fitur = next(iter(clients.values()))['X_train_t'].shape[1]
assert n_fitur == len(feature_names), "Jumlah fitur tidak cocok dengan nama fitur"
 
print(f"\nJumlah fitur  : {n_fitur}")
for i, nama in enumerate(feature_names, 1):
    print(f"{i:>3}. {nama}")
 
# ============================================================
# 4. RINGKASAN TIAP CLIENT
# ============================================================
rows = {}
for st, c in clients.items():
    rows[st] = {
        'n_train': len(c['y_train_t']),
        'n_val': len(c['y_val_t']),
        'n_test': len(c['y_test_t']),
        'pos_train': int(c['y_train_t'].sum()),
        'pos_val': int(c['y_val_t'].sum()),
        'pos_test': int(c['y_test_t'].sum()),
    }
 
ringkas = pd.DataFrame(rows).T
ringkas.index.name = 'state'
 
for s in ('train', 'val', 'test'):
    ringkas[f'pos_{s}_%'] = (100 * ringkas[f'pos_{s}'] / ringkas[f'n_{s}']).round(2)
 
total = ringkas[[c for c in ringkas.columns if not c.endswith('%')]].sum()
for s in ('train', 'val', 'test'):
    total[f'pos_{s}_%'] = round(100 * total[f'pos_{s}'] / total[f'n_{s}'], 2)
ringkas.loc['TOTAL'] = total
 
print("\nRingkasan per client:")
print(ringkas.to_string())
 
 
# %%
# ============================================================
# 5. clients -> fed_data
# ============================================================
fed_data = {}
for st, c in clients.items():
    fed_data[f"state_{st}"] = {
        "X_train": c["X_train_t"], "y_train": c["y_train_t"],
        "X_val": c["X_val_t"], "y_val": c["y_val_t"],
        "X_test": c["X_test_t"], "y_test": c["y_test_t"],
        "n_features": c["X_train_t"].shape[1],
    }
 
print("Client:", list(fed_data.keys()))
print("n_features per client:", {k: v["n_features"] for k, v in fed_data.items()})
 
# %%
# ============================================================
# 6. IMPOR RUNNER  (file: fl_runner.py)
# ============================================================
import fl_running1
importlib.reload(fl_running1)
from fl_running1 import (
    METRIC_KEYS, POOLED,
    run_federated,
    print_comparison,
    export_history_csv, export_per_client_csv,
    export_comparison_csv, export_rounds_comparison_csv,
    export_curves_csv, export_threshold_sweep_csv,
    plot_diagnostic_curves, plot_comparison_curves,
)
 
COMMON_DIM = 64
N_CLASSES = 2


Jumlah fitur  : 69
  1. sexvar
  2. _age80
  3. _educag
  4. income3
  5. veteran3
  6. children
  7. _urbstat
  8. _bmi5
  9. diabete4
 10. medcost1
 11. checkup1
 12. genhlth
 13. physhlth
 14. menthlth
 15. exerany2
 16. cvdstrk3
 17. chcscnc1
 18. chcocnc1
 19. chccopd3
 20. addepev3
 21. chckdny2
 22. havarth4
 23. diabage4
 24. deaf
 25. blind
 26. decide
 27. diffwalk
 28. diffdres
 29. diffalon
 30. _rfbing6
 31. drnkany6
 32. flushot7
 33. pneuvac4
 34. marital_1
 35. marital_2
 36. marital_3
 37. marital_4
 38. marital_5
 39. marital_6
 40. employ1_1
 41. employ1_2
 42. employ1_3
 43. employ1_4
 44. employ1_5
 45. employ1_6
 46. employ1_7
 47. employ1_8
 48. renthom1_1
 49. renthom1_2
 50. renthom1_3
 51. _imprace_1
 52. _imprace_2
 53. _imprace_3
 54. _imprace_4
 55. _imprace_5
 56. _imprace_6
 57. persdoc3_1
 58. persdoc3_2
 59. persdoc3_3
 60. _asthms1_1
 61. _asthms1_2
 62. _asthms1_3
 63. _smoker3_1
 64. _smoker3_2
 65. _smoker3_3
 66. _smoker3_4
 67. usenow3_1
 68. use

2026-10-23 16:47:05,101	INFO util.py:154 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.


In [4]:

# %%
# ============================================================
# 6a. HELPER PROGRESS (tqdm) + RINGKASAN ANTAR SEED
# ============================================================
def round_bar(total, desc, position=0, leave=True):
    """Buat progress bar per ronde + callback `on_round` untuk run_federated.
    Callback dipanggil fl_runner setiap selesai evaluasi satu ronde."""
    bar = tqdm(total=total, desc=desc, position=position, leave=leave,
               unit="ronde", dynamic_ncols=True)
 
    def on_round(rnd, n_rounds, loss, m):
        bar.update(1)
        bar.set_postfix(loss=f"{loss:.3f}", f1=f"{m['f1']:.3f}",
                        auc=f"{m['auc']:.3f}", ap=f"{m['ap']:.3f}")
 
    return bar, on_round
 
 
def _reset_ray(wait=8):
    """Matikan sesi Ray yang menggantung, lalu beri jeda agar port/proses bebas."""
    try:
        import ray
        if ray.is_initialized():
            ray.shutdown()
    except Exception:
        pass
    gc.collect()
    time.sleep(wait)
 
 
RUN_CACHE = OUT_DIR / "runs"
RUN_CACHE.mkdir(exist_ok=True)
RESUME = True      # True: lewati run yang sudah selesai (dibaca dari <OUT_DIR>/runs/*.pkl)
 
 
def _cache_path(tag, kw):
    """Kunci cache = semua parameter run + daftar client yang dipakai,
    supaya run top10 / top20 / all tidak saling menimpa."""
    items = [(k, repr(v)) for k, v in kw.items()
             if k not in ("on_round", "fed_data")]
    items.append(("clients", repr(sorted(kw["fed_data"].keys()))))
    key = hashlib.md5(repr(sorted(items)).encode()).hexdigest()[:10]
    return RUN_CACHE / f"{tag}_{key}.pkl"
 
 
def run_one(tag, bar, retries=3, **kw):
    """Satu run_federated dengan (a) cache hasil, (b) retry bila Ray gagal start
    atau simulasi tidak lengkap (mis. TimeoutError node Ray di Windows)."""
    cp = _cache_path(tag, kw)
    if RESUME and cp.exists():
        with open(cp, "rb") as f:
            h = pickle.load(f)
        bar.update(kw["num_rounds"] - bar.n)
        return h
 
    last = None
    for attempt in range(1, retries + 1):
        bar.reset()
        try:
            h = run_quiet(run_federated, **kw)
            if len(h["round"]) >= kw["num_rounds"] and h.get("test_best"):
                with open(cp, "wb") as f:
                    pickle.dump(h, f)
                return h
            last = RuntimeError(f"simulasi tidak lengkap "
                                f"({len(h['round'])}/{kw['num_rounds']} ronde)")
        except Exception as e:                      # noqa: BLE001
            last = e
        tqdm.write(f"[retry {attempt}/{retries}] {tag} gagal: "
                   f"{type(last).__name__}: {str(last)[:120]}")
        _reset_ray(wait=8 * attempt)
    raise RuntimeError(f"{tag} gagal setelah {retries} percobaan") from last
 
 
def run_grid(fed_data, modes, aggs, seeds, num_rounds, **run_kwargs):
    """Jalankan semua kombinasi mode x agregasi x seed dengan progress bar
    dua tingkat (total run + ronde). Hasil berbentuk sama seperti
    run_comparison: all_hist[mode][agg] = history seed pertama + ["runs"]."""
    total = len(modes) * len(aggs) * len(seeds)
    outer = tqdm(total=total, desc="Total run", position=0, unit="run",
                 dynamic_ncols=True)
    all_hist = {}
    try:
        for mode in modes:
            all_hist[mode] = {}
            for agg in aggs:
                runs = []
                for s in seeds:
                    outer.set_postfix_str(f"{mode}+{agg} seed={s}")
                    bar, cb = round_bar(num_rounds, f"  {mode}+{agg} s={s}",
                                        position=1, leave=False)
                    try:
                        h = run_one(
                            f"{mode}_{agg}_s{s}", bar,
                            fed_data=fed_data, name="cmp", common_dim=COMMON_DIM,
                            aggregation=agg, seed=s, mode=mode,
                            num_rounds=num_rounds, on_round=cb,
                            verbose_report=False, **run_kwargs)
                    finally:
                        bar.close()
                    runs.append(h)
                    outer.update(1)
                h0 = dict(runs[0])          # salinan dangkal (tanpa referensi melingkar)
                h0["runs"] = runs
                all_hist[mode][agg] = h0
    finally:
        outer.close()
    return all_hist
 
 
def seed_summary(histories, split="test", ddof=1):
    """{agg: {metrik: (mean, std)}} antar seed. split: 'test' | 'test_best'."""
    out = {}
    for agg, h in histories.items():
        runs = [r for r in h.get("runs", [h]) if r.get(split)]
        out[agg] = {}
        for k in METRIC_KEYS:
            v = np.array([r[split][k] for r in runs], dtype=float)
            v = v[~np.isnan(v)]
            if v.size == 0:
                out[agg][k] = (float("nan"), float("nan"))
            else:
                sd = float(v.std(ddof=ddof)) if v.size > ddof else float("nan")
                out[agg][k] = (float(v.mean()), sd)
    return out
 
 
def print_seed_summary(histories, split="test", ddof=1):
    summ = seed_summary(histories, split, ddof)
    tbl = {agg: {k: f"{mu:.4f} ± {sd:.4f}" for k, (mu, sd) in d.items()}
           for agg, d in summ.items()}
    print(f"\n=== Ringkasan antar seed ({split}, ddof={ddof}) ===")
    print(pd.DataFrame(tbl).T.to_string())
 
 
def export_seed_summary_csv(histories, out_path, split="test_best", ddof=1):
    summ = seed_summary(histories, split, ddof)
    with open(out_path, "w", newline="", encoding="utf-8") as f:
        w = csv.writer(f)
        w.writerow(["aggregation", *[f"{k}_{s}" for k in METRIC_KEYS
                                      for s in ("mean", "std")]])
        for agg, d in summ.items():
            w.writerow([agg, *[v for k in METRIC_KEYS for v in d[k]]])
    print(f"[export] ringkasan antar seed tersimpan di: {out_path}")
    return out_path
 
 
def export_seed_runs_csv(histories, out_path, split="test_best"):
    with open(out_path, "w", newline="", encoding="utf-8") as f:
        w = csv.writer(f)
        w.writerow(["aggregation", "seed", "best_round", *METRIC_KEYS])
        for agg, h in histories.items():
            for r in h.get("runs", [h]):
                if r.get(split):
                    w.writerow([agg, r["seed"], r.get("best_round"),
                                *[r[split][k] for k in METRIC_KEYS]])
    print(f"[export] hasil tiap seed tersimpan di: {out_path}")
    return out_path

In [5]:

# %%
# ============================================================
# 6b. HELPER DIAGNOSTIK (AUC-ROC, AP, kurva ROC/PR)
#     AUC/AP dan kurva ROC/PR sudah dihitung di fl_runner pada data test
#     (history[...]["test_best_per_client"] dan ["test_best_curves"]),
#     jadi TIDAK perlu model atau pelatihan ulang.
#     Kurva rata-rata = macro-average antar client (POOLED dilewati).
# ============================================================
GRID = np.linspace(0, 1, 201)
 
 
def _safe(fn, y, p):
    """NaN bila hanya satu kelas."""
    return float(fn(y, p)) if 0 < y.sum() < len(y) else float("nan")
 
 
def diagnostic_metrics(ys, ps):
    """AUC-ROC, AP, PR-AUC (trapesium) gabungan + per state + titik kurva.
    Dipakai oleh collect_result (bagian 13) yang punya probabilitas mentah."""
    y_all = np.concatenate(list(ys.values()))
    p_all = np.concatenate(list(ps.values()))
    per_state = pd.DataFrame({
        st: {"auc": _safe(roc_auc_score, ys[st], ps[st]),
             "ap": _safe(average_precision_score, ys[st], ps[st]),
             "prevalensi": float(ys[st].mean())}
        for st in ys}).T
    per_state.index.name = "state"
    fpr, tpr, _ = roc_curve(y_all, p_all)
    prec, rec, _ = precision_recall_curve(y_all, p_all)
    return {
        "auc": _safe(roc_auc_score, y_all, p_all),
        "ap": _safe(average_precision_score, y_all, p_all),
        "prauc": float(sk_auc(rec, prec)) if len(rec) > 1 else float("nan"),
        "prevalensi": float(y_all.mean()),
        "per_state": per_state,
        "roc": (fpr, tpr),
        "pr": (rec, prec),
    }
 
 
def macro_curve(curves, kind):
    """Rata-rata kurva antar client pada grid seragam.
    curves : history["test_best_curves"] (dict client -> kurva fl_runner).
    kind   : 'roc' | 'pr'. Return None bila tak ada client yang punya kurva."""
    ys = []
    for name, c in (curves or {}).items():
        if name == POOLED:
            continue
        if kind == "roc" and c.get("fpr"):
            ys.append(np.interp(GRID, c["fpr"], c["tpr"]))
        elif kind == "pr" and c.get("recall"):
            rec = np.asarray(c["recall"])[::-1]      # recall menurun -> naik
            prec = np.asarray(c["precision"])[::-1]
            ys.append(np.interp(GRID, rec, prec))
    return np.mean(ys, axis=0) if ys else None
 
 
def collect_seed_diagnostics(all_hist, split_mean="test_best",
                             split_pc="test_best_per_client",
                             split_curves="test_best_curves"):
    """Kumpulkan (seed-level, per-state, kurva) dari all_hist tanpa training ulang."""
    seed_rows, state_rows, curves = [], [], {}
    for mode, hs in all_hist.items():
        for agg, h in hs.items():
            for r in h.get("runs", [h]):
                m, pc = r.get(split_mean), r.get(split_pc)
                if not m or not pc:
                    continue
                prev = float(np.average(
                    [v.get("prevalence", np.nan) for v in pc.values()],
                    weights=[v["n"] for v in pc.values()]))
                seed_rows.append({
                    "mode": mode, "agg": agg, "seed": r["seed"],
                    "best_round": r.get("best_round"),
                    **{k: m[k] for k in METRIC_KEYS}, "prevalensi": prev})
                for cn, v in pc.items():
                    state_rows.append({
                        "mode": mode, "agg": agg, "seed": r["seed"], "state": cn,
                        "auc": v["auc"], "ap": v["ap"],
                        "prevalensi": v.get("prevalence", np.nan)})
                cv = r.get(split_curves)
                roc, pr = macro_curve(cv, "roc"), macro_curve(cv, "pr")
                if roc is not None and pr is not None:
                    curves.setdefault((mode, agg), []).append((roc, pr))
    return pd.DataFrame(seed_rows), pd.DataFrame(state_rows), curves
 
 
# %%
# ============================================================
# 7. TES CEPAT (sanity check)
# ============================================================
bar, cb = round_bar(2, "smoke")
try:
    h_test = run_quiet(
        run_federated,
        fed_data, name="smoke",
        common_dim=COMMON_DIM, n_classes=N_CLASSES,
        num_rounds=2, local_epochs=1,
        mode="sim", aggregation="fedavg",
        seed=SINGLE_SEED,
        verbose_report=False, on_round=cb, **THR_KW,
    )
finally:
    bar.close()
print(h_test["f1"], h_test["ap"], h_test["comm"])
 
# %%
# ============================================================
# 8. SATU KONFIGURASI PENUH
# ============================================================
FULL_ROUNDS = 10
bar, cb = round_bar(FULL_ROUNDS, "exp alt+pairwise")
try:
    history = run_quiet(
        run_federated,
        fed_data, name="exp",
        common_dim=COMMON_DIM, n_classes=N_CLASSES,
        num_rounds=FULL_ROUNDS, local_epochs=10, batch_size=64, lr=1e-3,
        mode="alt", aggregation="pairwise",
        epochs_v=3, epochs_u=2,
        select_by="f1",
        seed=SINGLE_SEED,
        verbose_report=False, on_round=cb, **THR_KW,
    )
finally:
    bar.close()
 
# laporan lengkap dicetak ke layar (bukan ke log)
from fl_running1 import report_results
history["summary"] = report_results(history, client_label="state")
 
print("\nTEST ronde terakhir (rata-rata client):", history["test"])
print("TEST ronde terbaik  (rata-rata client):", history.get("test_best"),
      "| best_round =", history.get("best_round"))
 
# kurva ROC/PR per client + pooled untuk run tunggal
plot_diagnostic_curves(history, str(OUT_DIR / "roc_pr_single.png"),
                       split="test_best_curves", title="alt+pairwise (ronde terbaik)")
 
# %%
# ============================================================
# 9. BANDINGKAN mode x aggregation (multi-seed)
#    Seed diatur di bagian 1 (SEEDS). Default 5 seed x 3 agregasi x 2 mode = 30 run.
#    Progress: bar "Total run" (30 run) + bar ronde per run.
#    Print dialihkan ke <OUT_DIR>/prints.log (set VERBOSE=True untuk melihat di layar).
# ============================================================
NUM_ROUNDS = 10
LOCAL_EPOCHS = 10
AGGS = ("fedavg", "fedprox", "pairwise")
MODES = ("sim", "alt")
 
all_hist = run_grid(
    fed_data, MODES, AGGS, SEEDS, NUM_ROUNDS,
    n_classes=N_CLASSES, local_epochs=LOCAL_EPOCHS,
    fedprox_mu=0.01, select_by="f1", **THR_KW,
)
 
 
def _fmt(ms):
    mu, sd = ms
    return f"{mu:.4f} ± {sd:.4f}"
 
 
rows = []
for mode, hs in all_hist.items():
    for agg, h in hs.items():
        runs = h.get("runs", [h])
        st = seed_summary({agg: h}, "test", ddof=1)[agg]
        sb = seed_summary({agg: h}, "test_best", ddof=1)[agg]
        row = {
            "config": f"{mode}+{agg}",
            "n_seeds": len(runs),
            "val_f1_last": np.nanmean([r["f1"][-1] for r in runs if r["f1"]]),
            "best_round": np.nanmean([r.get("best_round") for r in runs
                                      if r.get("best_round") is not None]),
            "total_MB": np.mean([r["comm"]["total_mb"] for r in runs]),
        }
        for k in ("f1", "auc", "ap"):
            row[f"test_{k}"] = _fmt(st[k])
        for k in METRIC_KEYS:
            row[f"testbest_{k}"] = _fmt(sb[k])
        rows.append(row)
summary = pd.DataFrame(rows).set_index("config")
print("\n", summary.round(4).to_string())
 
# %%
# ============================================================
# 9b. DIAGNOSTIK: AUC-ROC, AP, kurva ROC & PR (mean ± std antar seed)
#     Memakai hasil all_hist (tanpa training ulang).
# ============================================================
seed_df, state_df, curves = collect_seed_diagnostics(all_hist)
 
if not seed_df.empty:
    seed_df.round(4).to_csv(OUT_DIR / "diagnostic_per_seed.csv", index=False)
    state_df.round(4).to_csv(OUT_DIR / "diagnostic_per_state_per_seed.csv", index=False)
    print(seed_df.round(4).to_string(index=False))
 
    g = seed_df.groupby(["mode", "agg"])[["auc", "ap"]]
    mean, std = g.mean(), g.std()          # std: ddof=1
    diag_df = mean.round(4).astype(str) + " ± " + std.round(4).astype(str)
    diag_df["prevalensi"] = seed_df.groupby(["mode", "agg"])["prevalensi"].mean().round(4)
    diag_df.index = [f"{m}+{a}" for m, a in diag_df.index]
    diag_df.index.name = "config"
    print("\nAUC-ROC & AP, mean ± std antar seed (AP acak = prevalensi):\n",
          diag_df.to_string())
    diag_df.to_csv(OUT_DIR / "diagnostic_mean_std.csv")
 
    # ---- plot ROC & PR rata-rata antar seed ----
    fig, (a1, a2) = plt.subplots(1, 2, figsize=(13, 5))
    for (mode, agg), lst in curves.items():
        tprs = np.array([c[0] for c in lst])
        precs = np.array([c[1] for c in lst])
        lab = f"{mode}+{agg}"
        t_m, t_s = tprs.mean(0), tprs.std(0)
        p_m, p_s = precs.mean(0), precs.std(0)
        a1.plot(GRID, t_m, label=lab)
        a1.fill_between(GRID, t_m - t_s, t_m + t_s, alpha=.15)
        a2.plot(GRID, p_m, label=lab)
        a2.fill_between(GRID, p_m - p_s, p_m + p_s, alpha=.15)
 
    a1.plot([0, 1], [0, 1], "k--", alpha=.5)
    a1.set(title="ROC (macro antar client, rata-rata seed ± std)",
           xlabel="FPR", ylabel="TPR")
    base = seed_df["prevalensi"].mean()
    a2.axhline(base, color="k", ls="--", alpha=.5, label=f"acak (={base:.3f})")
    a2.set(title="Precision-Recall (macro antar client, rata-rata seed ± std)",
           xlabel="Recall", ylabel="Precision")
    for a in (a1, a2):
        a.grid(alpha=.3)
        a.legend(fontsize=7)
    plt.tight_layout()
    plt.savefig(OUT_DIR / "roc_pr_curves_seed.png", dpi=150)
    plt.show()
 
    # kurva POOLED (seed pertama) antar agregasi, per mode
    for mode, hs in all_hist.items():
        plot_comparison_curves(hs, str(OUT_DIR / f"{mode}_roc_pr_pooled.png"),
                               split="test_best_curves")
 
summary.to_csv(OUT_DIR / "summary.csv")
 
# %%
# ============================================================
# 10. DETAIL KOMPARASI PER CLIENT
# ============================================================
for mode, hs in all_hist.items():
    print(f"\n######## {mode} : ronde terbaik (val), seed pertama ########")
    print_comparison(hs, split="test_best_per_client", baseline="fedavg")
    if len(SEEDS) > 1:
        print_seed_summary(hs, split="test_best", ddof=1)
 
# %%
# ============================================================
# 11. EKSPOR CSV
# ============================================================
for mode, hs in tqdm(all_hist.items(), desc="Ekspor CSV", unit="mode"):
    for agg, h in hs.items():
        tag = f"{mode}_{agg}"
        export_history_csv(h, str(OUT_DIR / f"{tag}_rounds.csv"))
        export_per_client_csv(h, str(OUT_DIR / f"{tag}_per_client_val.csv"))
        export_per_client_csv(h, str(OUT_DIR / f"{tag}_per_client_test.csv"),
                              split="test_per_client")
        export_per_client_csv(h, str(OUT_DIR / f"{tag}_per_client_test_best.csv"),
                              split="test_best_per_client")
        export_curves_csv(h, str(OUT_DIR / f"{tag}_curves.csv"),
                          split="test_best_curves")
        export_threshold_sweep_csv(h, str(OUT_DIR / f"{tag}_threshold_sweep.csv"),
                                   split="test_best_sweep")
    export_comparison_csv(hs, str(OUT_DIR / f"{mode}_compare_test_best.csv"),
                          split="test_best_per_client")
    export_rounds_comparison_csv(hs, str(OUT_DIR / f"{mode}_compare_rounds.csv"))
    export_seed_summary_csv(hs, str(OUT_DIR / f"{mode}_seed_summary.csv"),
                            split="test_best", ddof=1)
    export_seed_runs_csv(hs, str(OUT_DIR / f"{mode}_seed_runs.csv"),
                         split="test_best")
summary.to_csv(OUT_DIR / "summary.csv")
 
# %%
# ============================================================
# 12. PLOT KURVA F1 PER RONDE (mean ± std antar seed)
# ============================================================
fig, axes = plt.subplots(1, len(all_hist), figsize=(12, 4), sharey=True)
axes = np.atleast_1d(axes)
for ax, (mode, hs) in zip(axes, all_hist.items()):
    for agg, h in hs.items():
        runs = [r for r in h.get("runs", [h]) if r["f1"]]
        L = min(len(r["f1"]) for r in runs)
        arr = np.array([r["f1"][:L] for r in runs], dtype=float)
        x = runs[0]["round"][:L]
        m, s = np.nanmean(arr, 0), np.nanstd(arr, 0)
        ax.plot(x, m, marker="o", label=agg)
        ax.fill_between(x, m - s, m + s, alpha=.15)
    ax.set_title(f"mode = {mode}")
    ax.set_xlabel("Ronde")
    ax.grid(alpha=.3)
    ax.legend()
axes[0].set_ylabel("Val F1 (rata-rata client, mean ± std seed)")
plt.tight_layout()
plt.savefig(OUT_DIR / "f1_per_round.png", dpi=150)
plt.show()
 
# %%
# ============================================================
# 13. (OPSIONAL) collect_result UNTUK PIPELINE FLOWER LAMA
#     Butuh: fl, config, load_global_model, predict_proba_clients
#     (milik pipeline Flower Anda, bukan fl_runner). Lewati bila tidak dipakai.
# ============================================================
def collect_result(secs):
    model = load_global_model()                           # noqa: F821
    pooled = fl.evaluate_pooled(model, split="test")      # noqa: F821
    df_test = fl.evaluate_on_clients(model, split="test") # noqa: F821
    ronde = pd.read_csv(config.OUT_DIR / "ronde_log.csv").reset_index(drop=True)  # noqa: F821
    bw, _ = fl.bandwidth_report(verbose=False)            # noqa: F821
 
    ys, ps = predict_proba_clients(model, "test")         # noqa: F821
    diag = diagnostic_metrics(ys, ps)
 
    f1 = ronde["f1"].astype(float)
    val_akhir = float(f1.iloc[-1]) if len(f1) else float("nan")
    if f1.notna().any():
        best = f1.idxmax()
        val_best, best_round = float(f1[best]), int(ronde.loc[best, "round"])
    else:
        val_best, best_round = float("nan"), -1
 
    pooled_num = {k: float(v) for k, v in pooled.items() if np.isscalar(v)}
    row = {
        **pooled_num,
        "auc": diag["auc"],
        "ap": diag["ap"],
        "prauc": diag["prauc"],
        "prevalensi": diag["prevalensi"],
        "f1_macro_klien": float(df_test["f1"].mean()),
        "f1_std_klien": float(df_test["f1"].std()),
        "val_f1_akhir": val_akhir,
        "val_f1_terbaik": val_best,
        "ronde_terbaik": best_round,
        "total_MB": float(bw["total_MB"]),
        "waktu_s": round(secs, 1),
    }
    return row, df_test["f1"], ronde, diag

smoke: 100%|██████████| 2/2 [00:12<00:00,  6.37s/ronde, ap=0.394, auc=0.685, f1=0.000, loss=0.544]


[0.0, 0.0] [0.31387592354944, 0.3938897675792684] {'theta_size_mb': 0.0238037109375, 'total_up_mb': 0.47607421875, 'total_down_mb': 0.47607421875, 'total_mb': 0.9521484375, 'avg_per_round_mb': 0.47607421875, 'avg_per_client_mb': 0.09521484375, 'full_total_mb': 1.64581298828125, 'saving_pct': 42.147227888002966}


exp alt+pairwise: 100%|██████████| 10/10 [00:21<00:00,  2.11s/ronde, ap=0.382, auc=0.663, f1=0.385, loss=1.500]


[ronde 10] val F1 0.3849 | AUC 0.6630 | AP 0.3819 | bandwidth ronde 0.476 MB
=== Bandwidth komunikasi (yang benar-benar dikirim: theta) ===
Total downlink (server->client): 2.380 MB
Total uplink   (client->server): 2.380 MB
TOTAL                          : 4.761 MB
Rata-rata per ronde            : 0.476 MB
Rata-rata per client (total)   : 0.476 MB
[Pembanding] bila FULL parameter: 8.229 MB (hemat 42.15%)
 round  val_loss    acc   prec    rec     f1    auc     ap  n_client_fit  down_MB  up_MB  total_MB
     1    0.5453 0.7610 0.3668 0.0163 0.0283 0.6928 0.4047            10   0.2380 0.2380    0.4761
     2    0.5118 0.7355 0.4573 0.4236 0.4190 0.7308 0.4521            10   0.2380 0.2380    0.4761
     3    0.5244 0.7291 0.4434 0.5143 0.4744 0.7346 0.4558            10   0.2380 0.2380    0.4761
     4    0.5523 0.7263 0.4379 0.5037 0.4681 0.7285 0.4514            10   0.2380 0.2380    0.4761
     5    0.6085 0.7206 0.4296 0.5051 0.4636 0.7202 0.4422            10   0.2380 0.2380    0.476

Total run: 100%|██████████| 30/30 [23:50<00:00, 47.68s/run, alt+pairwise seed=3407]


...

Ekspor CSV:  50%|█████     | 1/2 [00:00<00:00,  6.47mode/s]

[export] metrik per-ronde tersimpan di: hasil_fl_top10\sim_fedavg_rounds.csv
[export] metrik per-client tersimpan di: hasil_fl_top10\sim_fedavg_per_client_val.csv
[export] metrik test_per_client tersimpan di: hasil_fl_top10\sim_fedavg_per_client_test.csv
[export] metrik test_best_per_client tersimpan di: hasil_fl_top10\sim_fedavg_per_client_test_best.csv
[export] titik kurva ROC/PR tersimpan di: hasil_fl_top10\sim_fedavg_curves.csv
[export] threshold sweep tersimpan di: hasil_fl_top10\sim_fedavg_threshold_sweep.csv
[export] metrik per-ronde tersimpan di: hasil_fl_top10\sim_fedprox_rounds.csv
[export] metrik per-client tersimpan di: hasil_fl_top10\sim_fedprox_per_client_val.csv
[export] metrik test_per_client tersimpan di: hasil_fl_top10\sim_fedprox_per_client_test.csv
[export] metrik test_best_per_client tersimpan di: hasil_fl_top10\sim_fedprox_per_client_test_best.csv
[export] titik kurva ROC/PR tersimpan di: hasil_fl_top10\sim_fedprox_curves.csv
[export] threshold sweep tersimpan di:

Ekspor CSV: 100%|██████████| 2/2 [00:00<00:00,  5.90mode/s]

[export] titik kurva ROC/PR tersimpan di: hasil_fl_top10\alt_fedavg_curves.csv
[export] threshold sweep tersimpan di: hasil_fl_top10\alt_fedavg_threshold_sweep.csv
[export] metrik per-ronde tersimpan di: hasil_fl_top10\alt_fedprox_rounds.csv
[export] metrik per-client tersimpan di: hasil_fl_top10\alt_fedprox_per_client_val.csv
[export] metrik test_per_client tersimpan di: hasil_fl_top10\alt_fedprox_per_client_test.csv
[export] metrik test_best_per_client tersimpan di: hasil_fl_top10\alt_fedprox_per_client_test_best.csv
[export] titik kurva ROC/PR tersimpan di: hasil_fl_top10\alt_fedprox_curves.csv
[export] threshold sweep tersimpan di: hasil_fl_top10\alt_fedprox_threshold_sweep.csv
[export] metrik per-ronde tersimpan di: hasil_fl_top10\alt_pairwise_rounds.csv
[export] metrik per-client tersimpan di: hasil_fl_top10\alt_pairwise_per_client_val.csv
[export] metrik test_per_client tersimpan di: hasil_fl_top10\alt_pairwise_per_client_test.csv
[export] metrik test_best_per_client tersimpan d


C:\Users\s2\AppData\Local\Temp\ipykernel_24584\2889733215.py:300: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
